## Calculating the Units for Multi-family Homes across California

This notebook utilizes parcel data, Zillow data, and building data to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt

# statistical libraries
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

### Parcel data

The analysis relies on parcels to calculate which buildings are assigned to which units. Some buildings intersect with more than one parcel which is kept in both calculations would duplicate the volume of thse homes and cause erroneous unit calculations. Hence a unique parcel number is required to ensure that the calculations accurate.

In [2]:
# load parcels 
parcels = gpd.read_parquet("../../../../../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

In [3]:
# investigate duplicated parcel numbers
#parcels[parcels['PARNO'].duplicated()]


The parcel number should have been unique to each parcel. As shown above there are 525,130 parcel numbers that are not unique. Instead we'll assume that each row is a different parcel and assign a unique id from the index. 

In [4]:
# make an id column in the parcel dataframe to use as the unique id
parcels['parcel_ID'] = parcels.index

In [5]:
parcels.head()

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,parcel_ID
0,48-6298-3-2,Alameda,None,None,None,103.683187,606.327284,"MULTIPOLYGON Z (((-122.12384 37.75571 0, -122....",0
1,48-6313-23,Alameda,None,None,None,148.045063,1083.135315,"MULTIPOLYGON Z (((-122.12504 37.75429 0, -122....",1
2,48-6313-87,Alameda,None,None,None,97.520632,557.505860,"MULTIPOLYGON Z (((-122.12635 37.75366 0, -122....",2
3,48-6330-8-2,Alameda,None,None,None,171.271160,1602.682949,"MULTIPOLYGON Z (((-122.12243 37.75165 0, -122....",3
4,48-6432-32,Alameda,None,None,None,140.208009,1078.107621,"MULTIPOLYGON Z (((-122.12829 37.7611 0, -122.1...",4


### Zillow data

In [6]:
# import Zillow data 
zillow = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("../../../../../../capstone/electrigrid/data/ca_state_boundary.geojson").to_crs(epsg=4326)

# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

In [7]:
print(f"Number of Zillow points", zillow.shape[0])

Number of Zillow points 10012568


Handle zillow stacks:

- ALL NULL: keeps one NULL row

- One non-null: keeps non-null

- Conflicting non-null: keeps observation with max

In [12]:
zillow = (zillow
    .sort_values('unit', ascending=False, na_position='last')
    .drop_duplicates(subset='geometry', keep='first'))

In [14]:
print(f'Number of non-stacked points: {len(zillow)}')

Number of non-stacked points: 8779151


We expect to get the same number of homes at the end of unit regression calculation.

### Building Footprint data

Access: https://sat-io.earthengine.app/view/gba

In [15]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

# Unit-regression for multi-family homes and volume calculations for all homes

 Check that subsetting sums up to the total Zillow sum. 


In [16]:
# the analysis only requires the parcel_id and geometry  
parcels = parcels[['parcel_ID', 'geometry']]

## SUBSET ALL THE DATA TYPES
## select only multi-family data from Zillow
zillow_multi_raw = zillow[zillow['type'] == "Multi"]
zillow_multi_raw = zillow_multi_raw[zillow_multi_raw['code'] != "RR106"]

# select the single family home data 
zillow_single_raw = zillow[zillow['type'] == "Single"]

# select the condo data 
zillow_condos_raw = zillow[(zillow['code'] == "RR106") & (zillow['code'] != 'Single' )]

# check to ensure all of the subsets are accurate, these all add up to the total of the zillow 
assert zillow_multi_raw.shape[0] + zillow_single_raw.shape[0] + zillow_condos_raw.shape[0] == zillow.shape[0]

# set a zillow index column so we can track how much zillow data we lose
zillow_multi_raw['zillow_index'] = zillow_multi_raw.index

In [17]:
print(f"There are ",zillow_multi_raw.shape[0], "multi-family zillow.")

There are  591162 multi-family zillow.


#### Valid parcel subset

Subset to parcels with multi-family homes.

In [18]:
## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi_raw, predicate="contains")
## output: each parcel where the a multi_zillow point is within it (one-to-many relationship)

# confirm that joining with Zillow decreases the number of parcels
assert len(valid_parcels) < len(parcels)

# drop units as we will add back summed units per parcel after
valid_parcels = valid_parcels.drop(['unit', 'index_right'], axis = 1)

# sum number of units per parcel
units_sum = valid_parcels.sjoin(zillow_multi_raw, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
valid_parcels = valid_parcels.join(units_sum)
valid_parcels.rename(columns={"unit":"total_unit"}, inplace=True)
## for each parcel we should have the total number of units but this means that every time a parcel appears total units appear 

# drop duplicates based on parcel
valid_parcels = valid_parcels.drop_duplicates(subset = 'parcel_ID', keep = 'first')

In [19]:
print(f"We lose ", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_parcels['zillow_index'].unique()), "homes when assigning to parcels.")

We lose  11600 homes when assigning to parcels.


This could be new developments. The parcel data being from 2014 does not capture all of the current development. These missing homes could be new developments. 
**Assign these homes to their nearest neighbors to calculate their volumes.**

In [20]:
# find the multi-family homes that were not assigned to a parcel
multi_noparcel = zillow_multi_raw[~zillow_multi_raw.zillow_index.isin(valid_parcels['zillow_index'])]

len(multi_noparcel)

11600

As a reminder, `multi_noparcel` contains multi-family Zillow observations that are not within any of the parcels in our data.

In [21]:
# adjust to projected CRS for nearest join 
multi_noparcel = multi_noparcel.to_crs("EPSG:6933")
building = building.to_crs("EPSG:6933")

In [ ]:
# rename building index
# multi_noparcel = multi_noparcel.rename(columns = {'index__right':'building_index'})

In [22]:
multi_noparcel.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
5181231,Multi,NaN,NaN,None,None,None,502.0,240852282.0,living,332221.0,8654185,06075017604,651,PGE/SCE,RI112,POINT (-11811293.998 4485404.161),5181231
2713933,Multi,NaN,NaN,None,None,None,500.0,3972322.0,living,12672.0,3990955,06037408138,1290,PGE/SCE,RI109,POINT (-11376936.145 4095863.686),2713933
8931816,Multi,NaN,NaN,None,None,None,482.0,94522069.0,living,529249.0,4052724,06037123901,h2292,Others,RI107,POINT (-11423218.594 4112444.365),8931816
8282910,Multi,NaN,32.0,None,central,None,450.0,123989906.0,living,380276.0,2018507,06037143605,h2292,Others,RI104,POINT (-11420979.52 4108223.104),8282910
9577606,Multi,2005.0,NaN,None,None,None,450.0,21001322.0,living,149273.0,6664922,06067007103,h1257,Others,RI112,POINT (-11725374.691 4574207.972),9577606


In [23]:
# Join Zillow points to the nearest building geometry
multi_noparcel = gpd.sjoin_nearest(multi_noparcel,
                                        building,
                                        how="left", 
                                        distance_col='dist_to_building')

In [24]:
# check length after join to buildings
len(multi_noparcel)

13199

In [26]:
# drop bbox column -- the duplicated function doesn't like dictionaries
# multi_noparcel = multi_noparcel.drop('bbox', axis =1)

multi_noparcel.geometry.duplicated().sum()

1599

Duplicates created if buildings are equidistant from zillow points.

In [28]:
# drop duplicates
multi_noparcel = multi_noparcel.drop_duplicates(subset='geometry')

#### Buildings within multi-family parcels

Assign buildings to parcels which already includes Zillow data. 

In [29]:
# revert building CRS
building = building.to_crs('EPSG:4326')
multi_noparcel = multi_noparcel.to_crs('EPSG:4326')

In [30]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(valid_parcels, predicate="intersects")

# drop the unnecessary column 
valid_buildings = valid_buildings.drop(columns = 'index_right', axis = 1)

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)

In [31]:
valid_buildings.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-121.54641 40.08095, -121.54639 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-121.5463 40.08066, -121.5463 40.080...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
20391,ms,UnitedStates_023010010_21669,5.690192,0.455608,USA,"{'xmax': -121.9001822195566, 'xmin': -121.9002...","POLYGON ((-121.90024 40.48803, -121.90018 40.4...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0
20392,ms,UnitedStates_023010010_8153,8.068440,1.813950,USA,"{'xmax': -121.89911284380248, 'xmin': -121.899...","POLYGON ((-121.89923 40.48814, -121.89914 40.4...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0
20393,ms,UnitedStates_023010010_25824,6.629564,0.140848,USA,"{'xmax': -121.89948550755467, 'xmin': -121.899...","POLYGON ((-121.8996 40.48847, -121.89949 40.48...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0


In [32]:
print(f"We lose ", len(valid_parcels['parcel_ID'].unique()) - len(valid_buildings['parcel_ID'].unique()), "parcels when assigning buildings")

We lose  9497 parcels when assigning buildings


**Could there potentially be some parcels with no buildings? Or have missing building data?"**

In [33]:
print(f"From raw zillow to buidlings we lose", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()) ,"zillow points")
print(f"From parcels to buildings we lose", len(valid_parcels['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()), "zillow points")

From raw zillow to buidlings we lose 20876 zillow points
From parcels to buildings we lose 9276 zillow points


### Remove parcel duplicates

Buildings can straddle more than one parcel. This causes the volume to be utilized in the regression more than once. To avoid this we assign the building to the parcel where more of the building area is.

In [34]:
# change the crs to a projected crs
building_zillow_parcels = valid_buildings.to_crs("EPSG:6933")
valid_parcels_proj = valid_parcels.to_crs("EPSG:6933").set_index('parcel_ID')

In [35]:
len(building_zillow_parcels)

5485534

In [36]:
len(valid_parcels_proj)

610123

In [38]:
# to fix geometry problems from the crs change
building_zillow_parcels.geometry = building_zillow_parcels.geometry.buffer(0)
valid_parcels_proj.geometry = valid_parcels_proj.geometry.buffer(0)

In [39]:
# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(valid_parcels_proj.loc[building_zillow_parcels['parcel_ID']].geometry.values).area)

In [40]:
len(building_zillow_parcels)

5485534

In [41]:
# keep only the parcel with the largest overlap per building
building_zillow_parcels_merge = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .drop_duplicates(subset=["id", "parcel_ID"], keep="first") # fix -- keep per building, parcel pair
    .drop(columns="intersection_area"))

In [42]:
len(building_zillow_parcels_merge)

5464251

In theory, this value should be greater than the number of multi-family homes in the zillow data, and smaller than the number of `valid_buildings`.

In [43]:
len(building_zillow_parcels_merge) > len(zillow_multi_raw)

True

In [44]:
len(building_zillow_parcels_merge) < len(valid_buildings)

True

In [45]:
building_zillow_parcels = building_zillow_parcels.drop('bbox', axis =1)

building_zillow_parcels.duplicated().sum()

21280

In [46]:
building_zillow_parcels.head()

,source,id,height,var,region,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,intersection_area
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"POLYGON ((-11727561.053 4715034.549, -11727570...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,62.973518
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"POLYGON ((-11727550.366 4715005.976, -11727556...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,44.274674
20391,ms,UnitedStates_023010010_21669,5.690192,0.455608,USA,"POLYGON ((-11761700.693 4754873.402, -11761703...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,87.249637
20392,ms,UnitedStates_023010010_8153,8.068440,1.813950,USA,"POLYGON ((-11761602.803 4754883.715, -11761599...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,16.538912
20393,ms,UnitedStates_023010010_25824,6.629564,0.140848,USA,"POLYGON ((-11761638.944 4754916.097, -11761639...",10016366,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,134.804851


In [47]:
len(zillow_multi_raw)

591162

In [48]:
print(f"{len(building_zillow_parcels)} vs {len(building_zillow_parcels_merge)}")

5485534 vs 5464251


In [49]:
print(f"Difference that area method makes: {len(building_zillow_parcels) - len(building_zillow_parcels_merge)}")
print(f"Duplicates in original data: {building_zillow_parcels.duplicated().sum()}")

Difference that area method makes: 21283
Duplicates in original data: 21280


So our merge removes 5 buildings that were originally not duplicated...idk how this can be, but better than before!

In [50]:
print(f"We lose",len(valid_buildings['zillow_index'].unique()) - len(building_zillow_parcels['zillow_index'].unique()), "zillow points when trying to remove duplicates." )

We lose 0 zillow points when trying to remove duplicates.


### Calculate volume

**Using valid buildings since we lose so much data when trying to account for the duplicates**

The regression runs on the assumption that volume is linearly correlated with the number of units. First we calculate volume using the area of buildings. Then we run a regression to calculate the number of units where missing in the data.

We currently have three different data subsets:
- multi-family homes (assigned to building through parcels or nearest join)
- single-family homes
- condos

We will calculate volume for each.

#### Multi-family homes

We will concat the `multi_noparcel` and ` valid_buildings` data frames -- `multi_noparcel` will contains NA values in the parcel column.

In [51]:
# have to rename unit column to match "total_unit" in valid_buildings
multi_noparcel = multi_noparcel.rename(columns = {"unit":"total_unit"})

In [52]:
# and reproject data frame to crs with meters as units
building_m = pd.concat([valid_buildings, multi_noparcel]).to_crs("EPSG:6933") 

assert len(valid_buildings) + len(multi_noparcel) == len(building_m)

In [53]:
building_m.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index_right,dist_to_building
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-11727550.366 4715005.976, -11727550...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
20391,ms,UnitedStates_023010010_21669,5.690192,0.455608,USA,"{'xmax': -121.9001822195566, 'xmin': -121.9002...","POLYGON ((-11761700.693 4754873.402, -11761695...",10016366.0,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,NaN,NaN
20392,ms,UnitedStates_023010010_8153,8.068440,1.813950,USA,"{'xmax': -121.89911284380248, 'xmin': -121.899...","POLYGON ((-11761602.803 4754883.715, -11761594...",10016366.0,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,NaN,NaN
20393,ms,UnitedStates_023010010_25824,6.629564,0.140848,USA,"{'xmax': -121.89948550755467, 'xmin': -121.899...","POLYGON ((-11761638.944 4754916.097, -11761627...",10016366.0,Multi,NaN,NaN,None,None,None,1047500.0,None,NaN,9968621,06089012603,491,PGE/SCE,RI109,6268417,0.0,NaN,NaN


In [54]:
## CALCULATE VOLUME!

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

### Single-family homes

Join Zillow points to buildings for volume calculation.

In [55]:
# switch to projected CRS (for calculating area)
zillow_single = zillow_single_raw.to_crs("EPSG:6933")
building = building.to_crs("EPSG:6933")

# Join single family homes to nearest building (for area and height data)
zillow_single = zillow_single.sjoin(building, how = "left", predicate="within")

# Drop bbox column (creates error) and duplicates 
zillow_single = zillow_single.drop('bbox', axis =1 )
zillow_single = zillow_single.drop_duplicates()

# Save those unmatched to geometries
unmatched_single = zillow_single[zillow_single['index_right'].isna()]

# Remove those unmatched from data
zillow_single = zillow_single[~zillow_single['index_right'].isna()]

# rejoin building geometry
zillow_single['building_geometry'] = zillow_single['index_right'].map(building['geometry'][~building.index.duplicated()])

# set zillow geometry to building geo
zillow_single = zillow_single.set_geometry('building_geometry')

In [56]:
print(f"{len(zillow_single_raw)} vs {len(zillow_single) + len(unmatched_single)}")

7888172 vs 7888221


This join introduces new observations. The world will never know why.

#### Calculate volume!

In [57]:
# create column from polygon area
zillow_single['area_m2'] = zillow_single.geometry.area

# rename height column to be clear about units
zillow_single.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_single['volume_m3'] = zillow_single['area_m2'] * zillow_single['height_m']

In [58]:
len(zillow_single)

5708994

Concat observations with unmatched geometries (and therefore no volume), back to the data frame.

In [59]:
len(unmatched_single) + len(zillow_single)

7888221

In [60]:
unmatched_single = unmatched_single.rename(columns={"height":"height_m"})

In [61]:
zillow_single = pd.concat([zillow_single, unmatched_single])

In [62]:
len(zillow_single)

7888221

### Condos (same workflow as single family homes)

In [63]:
# switch to projected CRS (for calculating area)
zillow_condos = zillow_condos_raw.to_crs("EPSG:6933")

# Join condos to intersecting buildings (for area and height data)
zillow_condos = zillow_condos.sjoin(building, how = "left", predicate="within")

# Drop bbox column (creates error) and duplicates 
zillow_condos = zillow_condos.drop('bbox', axis =1 )
zillow_condos = zillow_condos.drop_duplicates()

# Save those unmatched to geometries
unmatched_condos = zillow_condos[zillow_condos['index_right'].isna()]

# Remove those unmatched from data
zillow_condos = zillow_condos[~zillow_condos['index_right'].isna()]

zillow_condos['building_geometry'] = zillow_condos['index_right'].map(building['geometry'][~building.index.duplicated()])

# set zillow geometry to building geo
zillow_condos = zillow_condos.set_geometry('building_geometry')

In [64]:
print(f"{len(zillow_condos_raw)} vs {len(zillow_condos) + len(unmatched_condos)}")

299817 vs 299818


And this join introduces 1 new condo observation! Again, we will never know why.

#### Calculate volume! For those with matched geometries

In [65]:
# create column from polygon area
zillow_condos['area_m2'] = zillow_condos.geometry.area

# rename height column to be clear about units
zillow_condos.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_condos['volume_m3'] = zillow_condos['area_m2'] * zillow_condos['height_m']

We will concat the data with unmatched geomtries (so no volume data) back to dataframe.

In [66]:
len(unmatched_condos) + len(zillow_condos)

299818

In [67]:
unmatched_condos = unmatched_condos.rename(columns={"height":"height_m"})

In [68]:
zillow_condos = pd.concat([zillow_condos, unmatched_condos])

In [69]:
len(zillow_condos)

299818

#### Units Assumption

We are making the assumption that the tax assessor data was correct in its identification of single-family homes and condos, we will assign the `unit` columns in both datasets to be exactly 1 for each observation.

In [70]:
zillow_single['total_unit'] = 1
zillow_condos['total_unit'] = 1

## The rest of the regression

This part of the workflow only works with the multi-family homes -- the ones that *have* unit data are being used to build the regression, and those with *missing* unit data use that regression to predict the number of units.

In [71]:
# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('parcel_ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'parcel_ID')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params.iloc[0]
slope = results_clean.params.iloc[1]

In [72]:
results_clean.params.iloc[1]

0.0009747352357970614

In [73]:
slope

0.0009747352357970614

In [74]:
intercept

5.606427090185207

#### Utilize the regression to estimate units where they're missing  

In [75]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) | (building_m['total_unit'] == 0)]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'parcel_ID')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

# utilize the values from the regression to calculate units
missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

# calculate each building's share of parcel volume
missing_outlier_units_pred['volume_share'] = (
    missing_outlier_units_pred['volume_m3'] / 
    missing_outlier_units_pred['missing_total_volume_m3'].replace(0, np.nan))

# distribute units proportionally, rounding to nearest int
missing_outlier_units_pred['total_unit'] = round(
    (intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope) 
    * missing_outlier_units_pred['volume_share'])

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual','total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

### Cap `total_unit` Value

For some outliers with high building volumes, our regression predicts an unreasonable number of units. We will cap our predictions at the 99.9th percentile of the non-predicted units. 

In [81]:
value = np.percentile(building_w_units['total_unit'], 99.9)

In [82]:
value

521.0

In [83]:
print(f"This would replace predicted unit values for {len(multi_complete[multi_complete['total_unit'] > value])} homes.")

This would replace predicted unit values for 461 homes.


In [84]:
# Set up if else statement
multi_complete['total_unit'] = np.where(multi_complete['total_unit'] > value, value, multi_complete['total_unit'])

### Create points from the building polygons

Single family homes and condos are points not polygons. For uniformity and hosting capacity calculation set the points as centroids to the parcels.

#### All multi-family homes

In [85]:
# projected CRS
multi_complete = multi_complete.to_crs("EPSG:6933")

# find centroid
multi_complete['centroid'] =  multi_complete.geometry.centroid

In [86]:
#### Single-family homes and condos

zillow_single['centroid'] = zillow_single.geometry.centroid

zillow_condos['centroid'] = zillow_condos.geometry.centroid

In [87]:
# If geometry for centroid is missing, replace with Zillow point
zillow_single['centroid'] = np.where(zillow_single['building_geometry'].isna(), zillow_single['geometry'], zillow_single['centroid'])
zillow_condos['centroid'] = np.where(zillow_condos['building_geometry'].isna(), zillow_condos['geometry'], zillow_condos['centroid'])

In [88]:
multi_complete.head()

,source,id,height_m,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index_right,dist_to_building,area_m2,volume_m3,total_volume_m3,volume_share,centroid
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN,62.973518,283.905634,853.757240,NaN,POINT (-11727564.987 4715038.471)
26238,osm,759299775,1.686291,0.644541,USA,"{'xmax': -121.43016199999998, 'xmin': -121.430...","POLYGON ((-11716354.676 4773227.596, -11716348...",10084995.0,Multi,NaN,1.0,None,None,None,448916.0,living,1064.0,9911222,06089012701,302,PGE/SCE,RI101,6245297,2.0,NaN,NaN,38.333495,64.641441,12930.381426,NaN,POINT (-11716349.663 4773227.774)
30005,ms,UnitedStates_023010010_16,3.319534,0.280181,USA,"{'xmax': -121.69706788384876, 'xmin': -121.697...","POLYGON ((-11742113.188 4792054.055, -11742112...",10001573.0,Multi,NaN,3.0,None,None,I,96001.0,living,1344.0,9910184,06089012701,357,PGE/SCE,RI101,6244425,1.0,NaN,NaN,283.595485,941.404872,7037.163187,NaN,POINT (-11742109.108 4792044.168)
31048,ms,UnitedStates_023010010_1720,7.291415,0.815212,USA,"{'xmax': -121.91076675749248, 'xmin': -121.910...","POLYGON ((-11762731.093 4796438.8, -11762721.4...",10086613.0,Multi,NaN,3.0,None,None,None,176731.0,living,1476.0,9908514,06089012606,411,PGE/SCE,RI101,6242951,3.0,NaN,NaN,91.268270,665.474852,7736.379409,NaN,POINT (-11762723.75 4796440.688)
31157,ms,UnitedStates_021232233_5134,6.432827,2.174101,USA,"{'xmax': -121.63049440002435, 'xmin': -121.630...","POLYGON ((-11735680.314 4804466.517, -11735681...",10000041.0,Multi,NaN,2.0,None,None,None,357000.0,living,3456.0,9908031,06089012701,623,PGE/SCE,RI101,6242607,4.0,NaN,NaN,316.484363,2035.889151,7581.420633,NaN,POINT (-11735686.517 4804476.272)


In [89]:
zillow_single.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height_m,var,region,building_geometry,area_m2,volume_m3,total_unit,centroid
883095,Single,2020.0,3.0,None,None,I,3040.0,896352.0,living,3040.0,1025756,06017030706,103,PGE/SCE,RR101,POINT (-11682529.894 4575729.112),3999102.0,ms,UnitedStates_023010211_584547,5.687031,1.345618,USA,"POLYGON ((-11415692.766 4085645.46, -11415692....",258.245732,1468.651551,1,POINT (-11415681.520937515 4085640.47515115)
2966995,Single,2003.0,4.0,fossil,central,None,1826.0,200834.0,living,1826.0,4310620,06047001801,36,PGE/SCE,RR101,POINT (-11622185.839 4440875.698),593756.0,ms,UnitedStates_023010320_13127,5.830075,0.935850,USA,"POLYGON ((-11849612.379 4945188.588, -11849605...",112.383509,655.204260,1,POINT (-11849616.99263067 4945190.522598983)
2967062,Single,2003.0,4.0,fossil,central,None,1826.0,198247.0,living,1826.0,4310687,06047001801,36,PGE/SCE,RR101,POINT (-11622253.091 4440887.725),593741.0,ms,UnitedStates_023010320_79199,5.970141,0.686197,USA,"POLYGON ((-11849626.012 4945258.104, -11849623...",79.594735,475.191783,1,POINT (-11849627.833278911 4945257.7879530685)
2967015,Single,2003.0,4.0,fossil,central,None,1826.0,212471.0,living,1826.0,4310640,06047001801,36,PGE/SCE,RR101,POINT (-11622170.787 4440812.808),593713.0,ms,UnitedStates_023010320_28594,4.969953,0.586084,USA,"POLYGON ((-11849476.733 4945263.316, -11849468...",132.112753,656.594179,1,POINT (-11849479.713382864 4945270.232543917)
2967013,Single,2003.0,4.0,fossil,central,None,1826.0,336848.0,living,1826.0,4310638,06047001801,36,PGE/SCE,RR101,POINT (-11622173.585 4440779.681),593719.0,ms,UnitedStates_023010320_81498,5.773458,0.551706,USA,"POLYGON ((-11849452.338 4945246.03, -11849436....",240.084131,1386.115764,1,POINT (-11849438.963974707 4945248.453164664)


In [90]:
zillow_condos.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height_m,var,region,building_geometry,area_m2,volume_m3,total_unit,centroid
5626224,Multi,NaN,NaN,None,None,None,1296.0,259208.0,living,490.0,9184924,06081603801,196,PGE/SCE,RR106,POINT (-11812408.343 4469155.358),2148872.0,osm,389843603,5.386683,2.066671,USA,"POLYGON ((-11427984.23 4123669.125, -11427989....",33.745923,181.778591,1,POINT (-11427988.50116544 4123668.8684305665)
5626110,Multi,NaN,NaN,None,None,O,1296.0,142309.0,living,490.0,9184810,06081603801,196,PGE/SCE,RR106,POINT (-11812646.858 4469410.867),2149168.0,osm,5831191,13.661510,13.415878,USA,"POLYGON ((-11427794.306 4123888.341, -11427786...",139.682217,1908.270073,1,POINT (-11427785.49813936 4123886.2203356097)
5626113,Multi,NaN,1.0,None,None,O,1296.0,233962.0,living,680.0,9184813,06081603801,196,PGE/SCE,RR106,POINT (-11812658.919 4469405.891),2149168.0,osm,5831191,13.661510,13.415878,USA,"POLYGON ((-11427794.306 4123888.341, -11427786...",139.682217,1908.270073,1,POINT (-11427785.49813936 4123886.2203356097)
5626004,Multi,NaN,NaN,None,None,O,1296.0,235035.0,living,490.0,9184704,06081603801,196,PGE/SCE,RR106,POINT (-11812652.358 4469408.632),2149168.0,osm,5831191,13.661510,13.415878,USA,"POLYGON ((-11427794.306 4123888.341, -11427786...",139.682217,1908.270073,1,POINT (-11427785.49813936 4123886.2203356097)
5626115,Multi,NaN,2.0,None,None,O,1296.0,280833.0,living,1030.0,9184815,06081603801,196,PGE/SCE,RR106,POINT (-11812668.182 4469402.336),2149168.0,osm,5831191,13.661510,13.415878,USA,"POLYGON ((-11427794.306 4123888.341, -11427786...",139.682217,1908.270073,1,POINT (-11427785.49813936 4123886.2203356097)


In [91]:
print(f"Our original zillow data had {len(zillow)}. We have a total of {len(multi_complete) + len(zillow_single) + len(zillow_condos)} observations in our current data. This is a difference of {(len(zillow) - (len(multi_complete) + len(zillow_single) + len(zillow_condos)))} observations.")

Our original zillow data had 8779151. We have a total of 9255778 observations in our current data. This is a difference of -476627 observations.


In [92]:
print(f"We lost {len(zillow_single_raw) - len(zillow_single)} observations in our single-family homes. This is most likely due to a lack of building footprint data.")

We lost -49 observations in our single-family homes. This is most likely due to a lack of building footprint data.


In [93]:
print(f"We lost {len(zillow_condos_raw) - len(zillow_condos)} observations in our condo data, most likely for similar reasons.")

We lost -1 observations in our condo data, most likely for similar reasons.


In [94]:
print(f"Our multi-family observations increased by {len(multi_complete) - len(zillow_multi_raw)}. This is due to the assignment of individual Zillow points to multiple building footprints.")

Our multi-family observations increased by 476577. This is due to the assignment of individual Zillow points to multiple building footprints.


## Final Step: Concatenate our three building types into one dataframe

In [129]:
final = pd.concat([multi_complete, zillow_single, zillow_condos])

In [130]:
print(type(final))
print(final.columns)
print(final.centroid.nunique())
print(final.crs)

<class 'geopandas.geodataframe.GeoDataFrame'>
Index(['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry',
       'parcel_ID', 'type', 'year', 'room', 'heat', 'cool', 'own', 'value',
       'sqft_type', 'sqft', 'ID', 'GEOID', 'p_ID', 'area', 'code',
       'zillow_index', 'total_unit', 'index_right', 'dist_to_building',
       'area_m2', 'volume_m3', 'total_volume_m3', 'volume_share', 'centroid',
       'unit', 'building_geometry'],
      dtype='object')
8887585
EPSG:6933


In [131]:
# Some cleaning up of columns
final = final.drop(['bbox', 'index_right', 'dist_to_building', 'geometry', 'building_geometry'], axis = 1)

In [132]:
print(type(final))

<class 'pandas.core.frame.DataFrame'>


In [133]:
final = gpd.GeoDataFrame(final, geometry='centroid', crs='EPSG:6933')

In [135]:
print(type(final))
print(final.crs)
print(final.columns)

<class 'geopandas.geodataframe.GeoDataFrame'>
EPSG:6933
Index(['source', 'id', 'height_m', 'var', 'region', 'parcel_ID', 'type',
       'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'sqft',
       'ID', 'GEOID', 'p_ID', 'area', 'code', 'zillow_index', 'total_unit',
       'area_m2', 'volume_m3', 'total_volume_m3', 'volume_share', 'centroid',
       'unit'],
      dtype='object')


In [136]:
final = final.set_geometry('centroid')

In [137]:
final = final.rename(columns={'centroid':'geometry'})

In [138]:
final = final.drop(columns='unit')
final = final.rename(columns={'total_unit':'unit'})

In [141]:
# Convert centroid column to parquet-legible geometry
# final['centroid'] = gpd.GeoSeries(final['centroid']).to_wkt()
final = final.set_geometry('geometry')

In [142]:
print(type(final))
print(final.crs)
print(final.columns)

<class 'geopandas.geodataframe.GeoDataFrame'>
EPSG:6933
Index(['source', 'id', 'height_m', 'var', 'region', 'parcel_ID', 'type',
       'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'sqft',
       'ID', 'GEOID', 'p_ID', 'area', 'code', 'zillow_index', 'unit',
       'area_m2', 'volume_m3', 'total_volume_m3', 'volume_share', 'geometry'],
      dtype='object')


In [144]:
final = final.to_crs(epsg=4326)
final.head(2)

,source,id,height_m,var,region,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,unit,area_m2,volume_m3,total_volume_m3,volume_share,geometry
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152.0,1.0,62.973518,283.905634,853.757240,NaN,POINT (-121.54645 40.08099)
26238,osm,759299775,1.686291,0.644541,USA,10084995.0,Multi,NaN,1.0,None,None,None,448916.0,living,1064.0,9911222,06089012701,302,PGE/SCE,RI101,6245297.0,2.0,38.333495,64.641441,12930.381426,NaN,POINT (-121.43021 40.6764)


In [145]:
# SAVE
final.to_parquet("~/../../capstone/electrigrid/data/final_building.parquet")

In [146]:
test_read = gpd.read_parquet("~/../../capstone/electrigrid/data/final_building.parquet")

In [147]:
len(test_read)

9255778

In [148]:
test_read.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [149]:
test_read.geometry

18573      POINT (-121.54645 40.08099)
26238       POINT (-121.43021 40.6764)
30005      POINT (-121.69719 40.87005)
31048      POINT (-121.91084 40.91538)
31157       POINT (-121.63062 40.9983)
                      ...             
1171202    POINT (-124.13049 40.57928)
1162668    POINT (-124.14605 40.76772)
1161700    POINT (-124.13446 40.79059)
1185930    POINT (-124.08289 40.89861)
1188126    POINT (-124.10033 40.93822)
Name: geometry, Length: 9255778, dtype: geometry